## **RAG Application using Type Sense**

In [1]:
import typesense

In [ ]:
client = typesense.Client({
    'nodes': [{
        'host': 'TYPESENSE_HOST', # Get from type sense cloud
        'port': '443', # For typesense cloud use 443
        'protocol': 'https'
    }],
    'api_key': 'TYPESENSE_API_KEY',
    'connection_timeout_seconds': 2

})

# Creating schema for our json data
books_schema = {
    'name': 'books',
    'fields': [
        {'name': 'title', 'type': 'string'},
        {'name': 'authors', 'type': 'string[]', 'facet': True},
        {'name': 'publication_year', 'type': 'int32', 'facet': True},
        {'name': 'ratings_count', 'type': 'int32'},
        {'name': 'average_rating', 'type': 'float'}
    ],
    'default_sorting_field': 'ratings_count'
}
print(client.collections.create(books_schema))

# After running go to that clusters tab >> go to >> collections
# there is one collection created as name books

Deprecation warning: AnalyticsRulesV1 is deprecated on v30+. Use client.analytics instead.


In [ ]:
client

In [ ]:
## This code will send the entire data or information to the collection named books
with open('books.jsonl', 'r', encoding='utf8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

Deprecation warning: Overrides is deprecated on v30+. Use client.curation_sets instead.
Deprecation warning: The synonyms API (collections/{collection}/synonyms) is deprecated is removed on v30+. Use synonym sets (synonym_sets) instead.


In [ ]:
search_parameters = {
    'q': "harry potter",
    'query_by': "title,authors",
    'sort_by': "ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441100921,
   'text_match_info': {'best_field_score': '2211897868288',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441100921',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

In [ ]:
search_parameters = {
    'q': 'harry potter',
    'query_by': 'title',
    'filter_by': 'publication_year:<1998',
    'sort_by': 'publication_year:desc'
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441100921,
   'text_match_info': {'best_field_score': '2211897868288',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441100921',
    'tokens_matched': 2,
    'typo_prefix_score': 0}}],
 'out_of': 9979,
 'page': 1,
 'request_params': {'collection_name

In [ ]:
search_parameters = {
    'q': "experiment",
    'query_by': 'title',
    'facet_by': 'authors',
    'sort_by': 'average_rating:desc'
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [{'counts': [{'count': 1,
     'highlighted': ' Käthe Mazur',
     'value': ' Käthe Mazur'},
    {'count': 1, 'highlighted': 'Mahatma Gandhi', 'value': 'Mahatma Gandhi'},
    {'count': 1, 'highlighted': 'Gretchen Rubin', 'value': 'Gretchen Rubin'},
    {'count': 1,
     'highlighted': 'James Patterson',
     'value': 'James Patterson'}],
   'field_name': 'authors',
   'sampled': False,
   'stats': {'total_values': 4}}],
 'found': 3,
 'hits': [{'document': {'authors': ['James Patterson'],
    'average_rating': 4.08,
    'id': '569',
    'image_url': 'https://images.gr-assets.com/books/1339277875m/13152.jpg',
    'publication_year': 2005,
    'ratings_count': 172302,
    'title': 'The Angel Experiment'},
   'highlight': {'title': {'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}],
   'text_match': 5787301

##### **Langchain + Typesense + Groq LLM + RAG Application**

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env into environment

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file")

In [ ]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("test.txt", encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

c:\Rag_Developement\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sujal warghe\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to

In [ ]:
docsearch = Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": "TYPESENSE_HOST",
        "port": "443",
        "protocol": "https",
        "typesense_api_key": "TYPESENSE_API_KEY",
        "typesense_collection_name": "lang-chain"
    },
)

Deprecation warning: Overrides is deprecated on v30+. Use client.curation_sets instead.
Deprecation warning: The synonyms API (collections/{collection}/synonyms) is deprecated is removed on v30+. Use synonym sets (synonym_sets) instead.


In [ ]:
query = "What is artificial intelligence"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Essay on Artificial Intelligence (AI)
Introduction
Artificial Intelligence (AI) refers to the ability of machines and computer systems to perform tasks that normally require human intelligence. These tasks include learning from experience, understanding natural language, recognizing patterns, solving problems, and making decisions. AI is one of the most transformative technologies of the 21st century and is rapidly changing the way humans live, work, and interact with the world. From smartphones and smart assistants to healthcare diagnostics and space exploration, AI has become an integral part of modern society.


In [ ]:
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x000001CB4F65BD10>, search_kwargs={})

In [ ]:
query = "Artificial intelligence indepth explanation"
retriever.invoke(query)[0]

Document(metadata={'source': 'test.txt'}, page_content='Essay on Artificial Intelligence (AI)\nIntroduction\nArtificial Intelligence (AI) refers to the ability of machines and computer systems to perform tasks that normally require human intelligence. These tasks include learning from experience, understanding natural language, recognizing patterns, solving problems, and making decisions. AI is one of the most transformative technologies of the 21st century and is rapidly changing the way humans live, work, and interact with the world. From smartphones and smart assistants to healthcare diagnostics and space exploration, AI has become an integral part of modern society.')